# Лабораторная работа 1
### Вариант 4

Выполнили:
> Галеев Егор Вячеславович, J3119, 505106

> Образцов Максим Евгеньевич, J3119, 466930

**Исходные данные**

- `housing.csv` - данные о ценах о жилье в Бостоне, где 
- `MEDV` - медианная стоимость жилья в (в тыс. $)
- Признаки (13 столбцов): криминальная обстановка, доля жилых зон, расстояние
до центров занятости, количество комнат и др.
- `test_size` = 0.27
- `random_state` = 40 
---

### 1. Цель работы 
Предложить среднеквадратическую аппроксимацию табличной функции многих переменных, проанализировать чувствительность точного решения к ошибкам округления,
проверить сходимость расчетных и исходных данных

### 2. Теоретические сведения

**Линейная регрессия** — это метод моделирования зависимости между зависимой переменной $y$ и одной или несколькими независимыми переменными $X$:

  $$y=Xw+\epsilon$$
  где $w$ — вектор весов (коэффициентов), а $\epsilon$ — случайная ошибка.  
**Задача обучения** заключается в поиске оптимальных весов $w$, которые минимизируют функцию потерь.
Для среднеквадратичной ошибки (MSE) решение записывается в виде:
$$w^{*}=\arg\min_{w}||Xw-y||^{2}$$

**Связь линейной регрессии с МНК**

Аналитическое решение задачи (МНК) сводится к решению системы линейных алгебраических уравнений (СЛАУ) 
  $$X^T X w = X^T y$$

**Методы предобработки данных**

- Масштабирование (`StandardScaler`): Приводит признаки к единому масштабу путем вычитания среднего (`with_mean`) и деления на стандартное отклонение (`with_std`). Если этого не сделать, признак с большими числовыми значениями, признак с большими числовыми значениями (например, налоги на дом в тысячах долларов) будет иметь больший вес при обучении, чем признак с маленькими значениями (например, количество комнат), хотя второй может быть важнее.
- Полиномиальные признаки (`PolynomialFeatures`): Создает новые признаки путем возведения исходных в степень `degree` и их перемножения. Это позволяет линейной модели выявлять нелинейные зависимости. Линейная регрессия сама по себе может строить только прямые линии (или гиперплоскости). Чтобы она могла улавливать сложные, нелинейные зависимости в данных, мы искусственно создаем новые признаки.

**Метрики качества**

- MAE (Mean Absolute Error): Средняя абсолютная ошибка, оценивающая разницу между предсказаниями и реальными значениями на обучающей (`train`) и тестовой (`test`) выборках.
- Число обусловленности: Вычисляется для матрицы $X^T X$ и показывает чувствительность точного решения к ошибкам округления. Большие значения указывают на проблемы.



In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 1.Загрузка данных
df = pd.read_csv('housing.csv', delimiter=';')

# Признаки X
X = df.drop(columns=['median_house_value']).fillna(0)
# Целевая переменная y
y = df['median_house_value']

# Параметры разбения данных (вариант 4)
test_size = 0.27
random_state = 40

# 2. Проведение экспериментов

# a) Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)



# Таблица экспериментов
experiments = [
    {'std': False, 'mean': False, 'degree': 1},
    {'std': True,  'mean': False, 'degree': 1},
    {'std': False, 'mean': True,  'degree': 1},
    {'std': True,  'mean': True,  'degree': 1},
    {'std': True,  'mean': True,  'degree': 2},
    {'std': True,  'mean': True,  'degree': 3},
]

results = []

for i, exp in enumerate(experiments, 1):
    # Инициализация препроцессоров
    scaler = StandardScaler(with_std=exp['std'], with_mean=exp['mean'])
    # include_bias=False запрещает добавлять в матрицу признаков нулевую степень (столбец из единиц)
    poly = PolynomialFeatures(degree=exp['degree'], include_bias=False)

    # b) Предобработка данных
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_poly = poly.fit_transform(X_train_scaled)
    X_test_poly = poly.transform(X_test_scaled)

    # c) Обучение модели
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    # Предсказание и оценка
    y_train_pred = model.predict(X_train_poly)
    y_test_pred = model.predict(X_test_poly)

    # d) Вычисление метрик
    # Вычисление MAE
    mae_train = mean_absolute_error(y_train, y_train_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)

    # Вычисление обусловленности матрицы X^T * X
    XT_X = X_train_poly.T @ X_train_poly
    cond_number = np.linalg.cond(XT_X)

    results.append({
        '№ эксперимента': i,
        'std': exp['std'],
        'mean': exp['mean'],
        'degree': exp['degree'],
        'MAE train': mae_train,
        'MAE test': mae_test,
        'Число обусловленности': f'{cond_number:.2e}',
    })

results_df = pd.DataFrame(results)
results_df

,№ эксперимента,std,mean,degree,MAE train,MAE test,Число обусловленности
0,1,False,False,1,50742.589260,51735.985379,8.77e+06
1,2,True,False,1,50742.589260,51735.985379,1.47e+05
2,3,False,True,1,50742.589260,51735.985379,2.51e+07
3,4,True,True,1,50742.589260,51735.985379,1.53e+02
4,5,True,True,2,45119.286940,45969.963526,2.21e+05
5,6,True,True,3,41494.556962,43637.394358,1.91e+09
